### Imports

In [ ]:
import csv
import heapq
from collections import defaultdict
import sys

###Description of class / fns:

        MCMF : class to find Min cost, Max flow
        
        fn __inti__: initializes adjacency list, graph, list of edges

        fn AE:
        Add an edge to the network
        u: from vertex
        v: to vertex
        capacity: edge capacity
        cost: cost per unit flow
        

        fn DWP:
        Find shortest path using Dijkstra with reduced costs
        Returns: (distance, parent, edge_used)
        

        fn mcmf:
        Compute minimum cost maximum flow using Primal-Dual algorithm
        src: source vertex
        sk: sink vertex
        Returns: (max_flow, min_cost)
      

In [ ]:


class MCMF:
    def __init__(self, n):
        self.n = n
        self.g = defaultdict(list)  # g means graph
        self.e = []  # e means edges

    def AE(self, u, v, capacity, cost):

        ex = len(self.e) # ex means edge index
        # For forward edge
        self.g[u].append(ex)
        self.e.append([v, capacity, cost, 0])  # [to, capacity, cost, flow]

        # For backward edge (residual)
        self.g[v].append(ex + 1)
        self.e.append([u, 0, -cost, 0])  # [to, capacity, cost, flow]

    def DWP(self, src, sk, ptl):   # ptl means potential, src means source, sk means sink

        d = [float('inf')] * (self.n + 1)  # d means distance
        p = [-1] * (self.n + 1)  # p means parent
        ue = [-1] * (self.n + 1)  # ue means used edge
        d[src] = 0

        pq = [(0, src)]

        while pq:
            d1, u = heapq.heappop(pq)

            if d1 > d[u]:
                continue

            if u == sk:
                break

            for ex in self.g[u]:
                v, cap, cost, f = self.e[ex] # f means flow


                if cap > f:

                    rc = cost + ptl[u] - ptl[v]  # rc means reduced cost
                    nd = d[u] + rc  # nd means new distance

                    if nd < d[v]:
                        d[v] = nd
                        p[v] = u
                        ue[v] = ex
                        heapq.heappush(pq, (nd, v))

        return d, p, ue

    def mcmf(self, src, sk):

        mf = 0   # mf is max flow
        mc = 0  # mc means min cost
        ptl = [0] * (self.n + 1)  # Dual variables (vertex potentials)

        itn = 0  # itn is iteration

        print("Starting Primal-Dual Algorithm for Min Cost Max Flow")
        print("=" * 60)

        while True:

            d, p, ue = self.DWP(src, sk, ptl)


            if d[sk] == float('inf'):
                break


            for i in range(1, self.n + 1):
                if d[i] < float('inf'):
                    ptl[i] += d[i]


            bk = float('inf') # bk means bottleneck
            v = sk
            path = []

            while v != src:
                u = p[v]
                eid = ue[v]
                path.append((u, v, eid))

                cap = self.e[eid][1]
                f = self.e[eid][3]
                bk = min(bk, cap - f)

                v = u

            path.reverse()


            pc = 0  # pc is path cost
            v = sk

            while v != src:
                u = p[v]
                eid = ue[v]


                self.e[eid][3] += bk

                self.e[eid ^ 1][3] -= bk

                pc += self.e[eid][2] * bk

                v = u

            mf += bk
            mc += pc

            itn += 1
            print(f"Iteration {itn}:")
            print(f"  Path found: {' -> '.join(str(u) for u, v, _ in path)} -> {sk}")
            print(f"  Bottleneck flow: {bk}")
            print(f"  Path cost: {pc}")
            print(f"  Total cost so far: {mc}")
            print(f"  Total flow so far: {mf}")
            print("-" * 60)

        print("=" * 60)
        print(f"Algorithm completed!")
        print(f"Maximum Flow: {mf}")
        print(f"Minimum Cost: {mc}")

        return mf, mc



### main function

In [ ]:
def main():
    file = open("Testcase9.csv", "r")

    try:
        rd = csv.reader(file)
        rw = list(rd)
        file.close()

        if len(rw) != 4:
            raise ValueError("CSV must have exactly 4 rows")

        fv = [int(x) for x in rw[0]]  # fv is the list showing 'from vertices'
        tv = [int(x) for x in rw[1]] # tv means list of 'to vertices'
        cts = [int(x) for x in rw[2]]
        costs = [int(x) for x in rw[3]]

        ne = len(fv)

        # Find maximum vertex id to determine network size
        mvx = max(max(fv), max(tv)) # mvx is max vertex

        # Create network
        nw = MCMF(mvx)

        # Adding edges
        for i in range(ne):
            nw.AE(fv[i], tv[i], cts[i], costs[i])

        # Src is vertex 1, sk is vertex 2
        src = 1
        sk = 2

        print(f"Source vertex: {src}")
        print(f"Sink vertex: {sk}\n")

        # Finding min cost max flow
        mf, mc = nw.mcmf(src, sk)

        return mf, mc

    except FileNotFoundError:
        print(f"Error: File not found")
        return None, None
    except Exception as e:
        print(f"Error: {e}")
        return None, None

if __name__ == "__main__":
    main()

Source vertex: 1
Sink vertex: 2

Starting Primal-Dual Algorithm for Min Cost Max Flow
Iteration 1:
  Path found: 1 -> 3 -> 4 -> 2
  Bottleneck flow: 2
  Path cost: 4
  Total cost so far: 4
  Total flow so far: 2
------------------------------------------------------------
Iteration 2:
  Path found: 1 -> 3 -> 2
  Bottleneck flow: 1
  Path cost: 4
  Total cost so far: 8
  Total flow so far: 3
------------------------------------------------------------
Iteration 3:
  Path found: 1 -> 4 -> 2
  Bottleneck flow: 1
  Path cost: 6
  Total cost so far: 14
  Total flow so far: 4
------------------------------------------------------------
Iteration 4:
  Path found: 1 -> 4 -> 3 -> 2
  Bottleneck flow: 1
  Path cost: 8
  Total cost so far: 22
  Total flow so far: 5
------------------------------------------------------------
Algorithm completed!
Maximum Flow: 5
Minimum Cost: 22
